In [17]:
import torch
from transformers import  CamembertForMaskedLM, CamembertTokenizer, CamembertModel,RobertaTokenizer,RobertaModel,RobertaForMaskedLM, pipeline
from transformers import BertForMaskedLM, BertTokenizer, BertModel, AlbertConfig, AlbertModel, AlbertTokenizer, AlbertForMaskedLM 
from transformers import TFAutoModel, AutoModelForMaskedLM, AutoTokenizer, AutoModelForCausalLM
import math
import numpy as np
import json
import random
import pandas as pd
import tensorflow as tf
from torch.nn import CrossEntropyLoss
from tqdm.notebook import tqdm

In [15]:
#Data
def prep_data(source):
    file = open(source, "r", encoding = 'utf-8')
    lines = file.readlines()

    data = []

    for l in range(12):
        line = lines[l]
        data.append(line)
    return data
data = prep_data("test/Polish.txt")

In [3]:
def see_tokens(model, tokenizer, data):
    for d in data:
        print(tokenizer.tokenize(d))

In [4]:
model = BertForMaskedLM.from_pretrained("dkleczek/bert-base-polish-uncased-v1",ignore_mismatched_sizes=True).cuda()
tokenizer = BertTokenizer.from_pretrained("dkleczek/bert-base-polish-uncased-v1")
see_tokens(model, tokenizer, data)

Some weights of the model checkpoint at dkleczek/bert-base-polish-uncased-v1 were not used when initializing BertForMaskedLM: ['cls.seq_relationship.weight', 'cls.seq_relationship.bias']
- This IS expected if you are initializing BertForMaskedLM from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing BertForMaskedLM from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).


['rodzice', ',', 'ktory', '##ch', 'dzieci', 'wykazu', '##ja', 'szczeg', '##ol', '##ne', 'zainteresowanie', 'jakim', '##s', 'rodzajem', 'sportu', ',', 'maja', 'do', 'podje', '##cia', 'trudna', 'decyzje', '.', 'czy', 'powinni', 'oni', 'pozwoli', '##c', 'swoim', 'dzieciom', 'na', 'tren', '##owanie', ',', 'by', '{', '}', 'sie', 'one', 'najlepszymi', 'sport', '##owcami', '?']
['dla', 'wielu', '{', '}', 'to', 'oznacza', 'rozpo', '##cze', '##cie', 'w', 'bardzo', 'młodym', 'wieku', '.']
['praca', 'w', 'szkole', ',', 'spotyka', '##nie', 'sie', 'ze', 'znajomymi', 'i', 'inne', 'zainteresowania', 'musza', 'zej', '##sc', 'na', '{', '}', 'plan', '.']
['bardzo', '{', '}', 'jest', 'wyjas', '##nic', 'małym', 'dzieciom', 'dlaczego', 'musza', 'tren', '##owac', 'piec', 'godzin', 'dziennie', '.']
['to', 'dotyczy', 'takze', 'weekend', '##ow', ',', 'kiedy', 'wiek', '##szo', '##sc', 'ich', 'przyjaci', '##oł', 'sie', '{', '}', '.']
['inny', 'problem', 'to', 'oczywiscie', 'pieniadze', '.', 'w', 'wielu', '{', '}

In [5]:
tokenizer = BertTokenizer.from_pretrained("UWB-AIR/Czert-A-base-uncased", from_tf=True)
model = AlbertForMaskedLM.from_pretrained("UWB-AIR/Czert-A-base-uncased", from_tf=True).cuda()
see_tokens(model, tokenizer, data)

The tokenizer class you load from this checkpoint is not the same type as the class this function is called from. It may result in unexpected tokenization. 
The tokenizer class you load from this checkpoint is 'AlbertTokenizer'. 
The class this function is called from is 'BertTokenizer'.
All TF 2.0 model weights were used when initializing AlbertForMaskedLM.

Some weights of AlbertForMaskedLM were not initialized from the TF 2.0 model and are newly initialized: ['predictions.decoder.weight', 'predictions.decoder.bias', 'predictions.decoder.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


['rod', '##zi', '##ce', ',', 'któ', '##rych', 'dzi', '##ec', '##i', 'wy', '##kazuj', '##ą', 'sz', '##cze', '##gó', '##ln', '##e', 'za', '##inter', '##es', '##ow', '##anie', 'jak', '##im', '##ś', 'rod', '##za', '##jem', 'sportu', ',', 'maj', '##ą', 'do', 'pod', '##ję', '##cia', 'tru', '##dn', '##ą', 'dec', '##y', '##z', '##ję', '.', 'czy', 'pow', '##in', '##ni', 'oni', 'poz', '##wo', '##li', '##ć', 'sw', '##o', '##im', 'dzi', '##ec', '##io', '##m', 'na', 'tren', '##ow', '##anie', ',', 'by', '{', '}', 'się', 'one', 'naj', '##lep', '##sz', '##y', '##mi', 'sport', '##ow', '##ca', '##mi', '?']
['dla', 'wie', '##lu', '{', '}', 'to', 'ozna', '##cz', '##a', 'rozpo', '##cz', '##ę', '##cie', 'w', 'bard', '##zo', 'm', '##ło', '##dy', '##m', 'wie', '##ku', '.']
['prac', '##a', 'w', 'sz', '##kol', '##e', ',', 'spot', '##yka', '##nie', 'się', 'ze', 'zna', '##jo', '##my', '##mi', 'i', 'inn', '##e', 'za', '##inter', '##es', '##ow', '##ania', 'mus', '##zą', 'zej', '##ść', 'na', '{', '}', 'plan', '.']
[

In [6]:
tokenizer = RobertaTokenizer.from_pretrained('gerulata/slovakbert')
model = RobertaForMaskedLM.from_pretrained('gerulata/slovakbert').cuda()
see_tokens(model, tokenizer, data)

['Rod', 'zi', 'ce', ',', 'Ġkt', 'Ã³', 'rych', 'Ġd', 'zie', 'ci', 'Ġw', 'y', 'kaz', 'uj', 'Äħ', 'Ġsz', 'c', 'ze', 'gÃ³', 'lne', 'Ġzainteres', 'ow', 'anie', 'Ġjak', 'im', 'ÅĽ', 'Ġrod', 'za', 'jem', 'Ġspor', 'tu', ',', 'Ġmaj', 'Äħ', 'Ġdo', 'Ġpod', 'j', 'ÄĻ', 'cia', 'Ġtru', 'd', 'n', 'Äħ', 'Ġde', 'cy', 'z', 'j', 'ÄĻ', '.', 'ĠC', 'zy', 'Ġpo', 'win', 'ni', 'Ġoni', 'Ġpoz', 'wo', 'li', 'Äĩ', 'Ġs', 'wo', 'im', 'Ġd', 'zie', 'ci', 'om', 'Ġna', 'Ġtr', 'eno', 'w', 'anie', ',', 'Ġby', 'Ġ', '{', '}', 'Ġsi', 'ÄĻ', 'Ġone', 'Ġnajlep', 's', 'zy', 'mi', 'Ġspor', 'to', 'w', 'cami', '?', 'Ġ', 'Ċ']
['D', 'la', 'Ġwie', 'lu', 'Ġ', '{', '}', 'Ġto', 'Ġozna', 'c', 'za', 'Ġrozpo', 'cz', 'ÄĻ', 'cie', 'Ġw', 'Ġb', 'ard', 'zo', 'Ġm', 'ÅĤ', 'od', 'ym', 'Ġwie', 'ku', '.', 'Ċ']
['Pra', 'ca', 'Ġw', 'Ġs', 'zko', 'le', ',', 'Ġspo', 'ty', 'kanie', 'Ġsi', 'ÄĻ', 'Ġze', 'Ġzna', 'jom', 'ymi', 'ĠI', 'Ġin', 'ne', 'Ġzainteres', 'ow', 'ania', 'Ġmu', 'sz', 'Äħ', 'Ġzej', 'ÅĽ', 'Äĩ', 'Ġna', 'Ġ', '{', '}', 'Ġplan', '.', 'Ċ']
['B', 'ard'

In [7]:
tokenizer = AutoTokenizer.from_pretrained("Milos/slovak-gpt-j-1.4B")
model = AutoModelForCausalLM.from_pretrained("Milos/slovak-gpt-j-1.4B").cuda()
see_tokens(model, tokenizer, data)

['Rod', 'zi', 'ce', ',', 'Ġk', 'tÃ³', 'rych', 'Ġd', 'zie', 'ci', 'Ġw', 'y', 'kaz', 'uj', 'Äħ', 'Ġs', 'z', 'c', 'ze', 'gÃ³', 'lne', 'Ġza', 'inte', 're', 'so', 'w', 'anie', 'Ġjak', 'im', 'ÅĽ', 'Ġrod', 'za', 'jem', 'Ġspo', 'rtu', ',', 'Ġmaj', 'Äħ', 'Ġdo', 'Ġpod', 'j', 'ÄĻ', 'cia', 'Ġtru', 'd', 'n', 'Äħ', 'Ġde', 'cy', 'z', 'j', 'ÄĻ', '.', 'ĠC', 'zy', 'Ġpo', 'win', 'ni', 'Ġoni', 'Ġpoz', 'wo', 'li', 'Äĩ', 'Ġs', 'wo', 'im', 'Ġd', 'zie', 'ci', 'om', 'Ġna', 'Ġtr', 'eno', 'w', 'anie', ',', 'Ġby', 'Ġ{', '}', 'Ġsi', 'ÄĻ', 'Ġone', 'Ġnajlep', 's', 'zy', 'mi', 'Ġspor', 'to', 'w', 'cami', '?', 'Ġ', 'Ċ']
['D', 'la', 'Ġw', 'ie', 'lu', 'Ġ{', '}', 'Ġto', 'Ġozna', 'cza', 'Ġrozpo', 'cz', 'ÄĻ', 'cie', 'Ġw', 'Ġb', 'ard', 'zo', 'Ġm', 'ÅĤ', 'od', 'ym', 'Ġw', 'ie', 'ku', '.', 'Ċ']
['Pra', 'ca', 'Ġw', 'Ġs', 'zko', 'le', ',', 'Ġspo', 'ty', 'kanie', 'Ġsi', 'ÄĻ', 'Ġze', 'Ġzna', 'jom', 'y', 'mi', 'ĠI', 'Ġin', 'ne', 'Ġza', 'inte', 're', 'so', 'w', 'ania', 'Ġmu', 'sz', 'Äħ', 'Ġz', 'ej', 'ÅĽ', 'Äĩ', 'Ġna', 'Ġ{', '}', 'Ġ

In [18]:
#pandas, tqdm
langs = ["Polish","Czech","Slovak"]

tokenizer_pl = BertTokenizer.from_pretrained("dkleczek/bert-base-polish-uncased-v1")
tokenizer_cz = BertTokenizer.from_pretrained("UWB-AIR/Czert-A-base-uncased", from_tf=True)
tokenizer_sl = RobertaTokenizer.from_pretrained('gerulata/slovakbert')
tokenizer_sl_gpt = AutoTokenizer.from_pretrained("Milos/slovak-gpt-j-1.4B")

for lang in tqdm(langs):
    print(lang)
    data = prep_data("test/"+lang+".txt")
    for d in data:
        print("PL:",len(tokenizer_pl.tokenize(d)),"CZ:",len(tokenizer_cz.tokenize(d)),"SL:",len(tokenizer_sl.tokenize(d)),"SL(GPT):",len(tokenizer_sl_gpt.tokenize(d)))

The tokenizer class you load from this checkpoint is not the same type as the class this function is called from. It may result in unexpected tokenization. 
The tokenizer class you load from this checkpoint is 'AlbertTokenizer'. 
The class this function is called from is 'BertTokenizer'.


  0%|          | 0/3 [00:00<?, ?it/s]

Polish
PL: 43 CZ: 80 SL: 90 SL(GPT): 93
PL: 14 CZ: 23 SL: 27 SL(GPT): 27
PL: 20 CZ: 33 SL: 35 SL(GPT): 39
PL: 16 CZ: 35 SL: 42 SL(GPT): 42
PL: 17 CZ: 26 SL: 33 SL(GPT): 32
PL: 24 CZ: 52 SL: 56 SL(GPT): 54
PL: 22 CZ: 38 SL: 41 SL(GPT): 42
PL: 20 CZ: 37 SL: 44 SL(GPT): 42
PL: 51 CZ: 102 SL: 108 SL(GPT): 109
PL: 21 CZ: 38 SL: 44 SL(GPT): 46
PL: 11 CZ: 19 SL: 23 SL(GPT): 24
PL: 27 CZ: 48 SL: 53 SL(GPT): 53
Czech
PL: 66 CZ: 46 SL: 53 SL(GPT): 61
PL: 14 CZ: 11 SL: 13 SL(GPT): 13
PL: 17 CZ: 11 SL: 17 SL(GPT): 18
PL: 26 CZ: 15 SL: 23 SL(GPT): 28
PL: 23 CZ: 15 SL: 21 SL(GPT): 21
PL: 40 CZ: 19 SL: 27 SL(GPT): 34
PL: 30 CZ: 19 SL: 24 SL(GPT): 30
PL: 35 CZ: 18 SL: 27 SL(GPT): 29
PL: 67 CZ: 39 SL: 54 SL(GPT): 63
PL: 32 CZ: 22 SL: 29 SL(GPT): 29
PL: 15 CZ: 14 SL: 16 SL(GPT): 16
PL: 36 CZ: 25 SL: 29 SL(GPT): 36
Slovak
PL: 69 CZ: 73 SL: 42 SL(GPT): 41
PL: 14 CZ: 13 SL: 12 SL(GPT): 11
PL: 16 CZ: 16 SL: 13 SL(GPT): 12
PL: 26 CZ: 30 SL: 18 SL(GPT): 17
PL: 24 CZ: 29 SL: 18 SL(GPT): 17
PL: 36 CZ: 32 SL: 22